In [ ]:
# ============================================================
# COMPETITIVE PACING – FULL PIPELINE + SIMPLE CLI (ONE CELL)
# - Cleans new stroke datasets
# - Trains & saves per-split fraction models per stroke
# - Provides a simple CLI with:
#     1) Pre-race ideal splits generator
#     2) Post-race given vs ideal analysis
#   and optional JPEG chart saving.
# ============================================================

import os, re, json
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ------------------------------------------------------------
# PATHS / GLOBALS
# ------------------------------------------------------------
ROOT = Path(".").resolve()
DATA_DIR = ROOT  # assumes CSVs are in current working dir
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

STROKE_FILES = {
    "Freestyle": DATA_DIR / "freestyle_dataset.csv",
    "Backstroke": DATA_DIR / "backstroke_dataset.csv",
    "Breaststroke": DATA_DIR / "breaststroke_dataset.csv",
    "Butterfly": DATA_DIR / "butterfly_dataset.csv",
    "IM": DATA_DIR / "im_dataset.csv",
}
STROKES = list(STROKE_FILES.keys())

# ------------------------------------------------------------
# GENERIC HELPERS (TIME + SPLITS)
# ------------------------------------------------------------
def parse_time_to_seconds(s: str) -> float:
    """
    Parse strings like:
      - '68.33' (seconds)
      - '261'   (seconds)
      - '1:02.97' (min:sec)
      - '2.02.97' (special mm.ss.xx pattern from dataset)
    Return seconds as float or NaN.
    """
    s = str(s).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    # colon format: mm:ss(.xx) or h:mm:ss(.xx)
    if ":" in s:
        parts = s.split(":")
        try:
            parts = [float(p) for p in parts]
        except:
            return np.nan
        secs = 0.0
        for p in parts:
            secs = secs * 60.0 + p
        return secs

    # special weird "2.02.97" format -> 2:02.97
    if s.count(".") > 1:
        segs = s.split(".")
        try:
            mins = int(segs[0])
            secs = int(segs[1])
            hundredths = int(segs[2])
            return mins * 60.0 + secs + hundredths / 100.0
        except:
            pass

    # plain numeric seconds
    try:
        return float(s)
    except:
        return np.nan


def seconds_to_time_str(x: float) -> str:
    """Convert seconds -> m:ss.xx string."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "NA"
    x = float(x)
    m = int(x // 60)
    s = x - 60 * m
    return f"{m}:{s:05.2f}"


def parse_splits_string(s: str) -> List[float]:
    """Extract numeric splits from a free-form ';'-style splits string."""
    nums = re.findall(r"[-+]?\d*\.?\d+", str(s))
    return [float(x) for x in nums] if nums else []


def print_splits(splits: List[float]) -> str:
    """Nicely formatted splits string."""
    return "; ".join(f"{v:.2f}" for v in splits)


# ------------------------------------------------------------
# DATA CLEANING
# ------------------------------------------------------------
def clean_stroke_df(stroke_label: str, path: Path) -> pd.DataFrame:
    """
    Load and clean a single stroke dataset:
    - Fix column names
    - Clean times & distances
    - Parse splits into lists
    - Standardize stroke/pool/strategy/age
    """
    df = pd.read_csv(path, encoding="latin1")

    rename_map = {}
    for c in df.columns:
        cl = c.strip()
        low = cl.lower()
        if low.startswith("final time in seconds"):
            rename_map[c] = "final_time_sec_raw"
        elif "splits per 50" in cl:
            rename_map[c] = "splits_raw"
        elif low.startswith("athlete age"):
            rename_map[c] = "athlete_age"
        elif low.startswith("im stroke"):
            rename_map[c] = "im_labels"
        elif low.startswith("full name"):
            rename_map[c] = "Full Name"
    if rename_map:
        df = df.rename(columns=rename_map)

    # Distance -> numeric
    if "Distance" in df.columns:
        df["Distance"] = pd.to_numeric(df["Distance"], errors="coerce")
    else:
        df["Distance"] = np.nan

    # Preserve raw stroke text but standardize main Stroke col
    df["Stroke_raw"] = df["Stroke"]
    df["Stroke"] = stroke_label

    # Clean pool & strategy text
    df["pool"] = df["pool"].astype(str).str.strip()
    df["Strategy"] = df["Strategy"].astype(str).str.strip()

    # Age
    if "athlete_age" in df.columns:
        df["athlete_age"] = pd.to_numeric(df["athlete_age"], errors="coerce")
    else:
        df["athlete_age"] = np.nan

    # Final time in seconds
    if "final_time_sec_raw" in df.columns:
        df["final_time_sec"] = df["final_time_sec_raw"].apply(parse_time_to_seconds)
    else:
        df["final_time_sec"] = np.nan

    # Splits
    if "splits_raw" in df.columns:
        df["splits_list"] = df["splits_raw"].apply(parse_splits_string)
    else:
        df["splits_list"] = [[] for _ in range(len(df))]
    df["num_splits"] = df["splits_list"].apply(len)

    return df


def add_fraction_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each row, create frac_k = split_k / final_time_sec.
    Adds frac_1 ... frac_max where max is max num_splits in df.
    """
    max_splits = int(df["num_splits"].max() or 0)
    for k in range(1, max_splits + 1):
        col = f"frac_{k}"
        vals = []
        for splits, T in zip(df["splits_list"], df["final_time_sec"]):
            if not isinstance(splits, list) or len(splits) < k or not pd.notna(T) or T <= 0:
                vals.append(np.nan)
            else:
                vals.append(float(splits[k - 1]) / float(T))
        df[col] = vals
    return df


# ------------------------------------------------------------
# MODEL TRAINING & SAVING
# ------------------------------------------------------------
def train_models_for_stroke(stroke_label: str, df: pd.DataFrame):
    """
    For a given stroke:
    - Add frac_k targets
    - Train one model per frac_k using:
        - Numeric: Distance, final_time_sec, athlete_age
        - Categorical: pool, Strategy
    - Save each pipeline and index.json for CLI lookup.
    """
    df = df.copy()
    # Keep only rows with a valid final time
    df = df[df["final_time_sec"].notna() & (df["final_time_sec"] > 0)]
    df = add_fraction_targets(df)

    max_splits = int(df["num_splits"].max() or 0)
    stroke_dir = MODELS_DIR / stroke_label
    stroke_dir.mkdir(parents=True, exist_ok=True)

    num_features = ["Distance", "final_time_sec", "athlete_age"]
    cat_features = ["pool", "Strategy"]

    num_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ]
    )

    index_meta = []

    for k in range(1, max_splits + 1):
        target_col = f"frac_{k}"
        df_k = df[df[target_col].notna()]
        if df_k.empty:
            continue

        X = df_k[num_features + cat_features]
        y = df_k[target_col].astype(float)

        # Model choice & params (aligned with previous leaderboard-style approach)
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=6,
            random_state=42,
        )

        pipe = Pipeline(
            [
                ("pre", pre),
                ("model", model),
            ]
        )

        pipe.fit(X, y)

        model_path = stroke_dir / f"{stroke_label}_frac_{k}.joblib"
        joblib.dump(pipe, model_path)

        index_meta.append(
            {
                "target": target_col,
                "model_path": str(model_path),
                "features": num_features + cat_features,
            }
        )

    index_path = stroke_dir / "index.json"
    with open(index_path, "w") as f:
        json.dump(index_meta, f, indent=2)

    return index_meta


def train_all_strokes():
    """Clean all stroke datasets and train everything."""
    all_clean = {}
    for stroke_label, path in STROKE_FILES.items():
        df_clean = clean_stroke_df(stroke_label, path)
        all_clean[stroke_label] = df_clean
        print(f"{stroke_label}: {len(df_clean)} rows, cleaned")
        train_models_for_stroke(stroke_label, df_clean)
    return all_clean


# ------------------------------------------------------------
# TRAIN MODELS (runs once when cell is executed)
# ------------------------------------------------------------
all_clean = train_all_strokes()
print("Training done. Models saved in:", MODELS_DIR)

# ------------------------------------------------------------
# CLI INFERENCE HELPERS (LOAD MODELS & PREDICT FRACTIONS)
# ------------------------------------------------------------
def expected_num_splits(distance: int) -> int:
    """50m-based splits count."""
    return distance // 50 if distance and distance > 0 else 0


def frac_name(i: int) -> str:
    return f"frac_{i}"


def load_index(stroke: str) -> List[Dict]:
    idx_path = MODELS_DIR / stroke / "index.json"
    if not idx_path.exists():
        raise FileNotFoundError(f"Missing index for {stroke}: {idx_path}")
    meta = json.loads(idx_path.read_text())
    # Normalize and sort by frac number
    meta = [m for m in meta if re.match(r"(?i)^frac(_)?\d+$", str(m.get("target", "")))]
    for m in meta:
        t = str(m["target"]).lower()
        if re.match(r"^frac\d+$", t):
            t = re.sub(r"^frac(\d+)$", r"frac_\1", t)
        m["target"] = t
    meta.sort(key=lambda d: int(re.findall(r"\d+", d["target"])[0]))
    return meta


def load_frac_models(stroke: str) -> Tuple[Dict[str, object], Dict[str, Dict]]:
    metas = load_index(stroke)
    models, meta_map = {}, {}
    for m in metas:
        p = Path(m["model_path"])
        if not p.exists():
            alt = MODELS_DIR / stroke / p.name
            if alt.exists():
                p = alt
        if not p.exists():
            raise FileNotFoundError(f"Model file missing: {p}")
        models[m["target"]] = joblib.load(p)
        meta_map[m["target"]] = m
    return models, meta_map


def build_feature_row(required_feats: List[str],
                      distance_m: int,
                      total_time_s: float) -> pd.DataFrame:
    """Single-row DF for prediction, leaving unknowns NaN (imputed in pipeline)."""
    row = {f: np.nan for f in required_feats}
    if "Distance" in row:
        row["Distance"] = float(distance_m)
    if "final_time_sec" in row:
        row["final_time_sec"] = float(total_time_s)
    if "Final time in seconds" in row:
        row["Final time in seconds"] = float(total_time_s)
    if "athlete_age" in row:
        row["athlete_age"] = np.nan  # unknown at inference by default
    if "pool" in row:
        row["pool"] = "LCM"          # default assumption; can be adjusted later
    if "Strategy" in row:
        row["Strategy"] = "Holding_pace"
    return pd.DataFrame([row])


def predict_fractions(stroke: str, distance_m: int, total_time_s: float) -> pd.Series:
    """
    Use trained models to predict per-50 fractions for a given stroke/distance/time.
    Fractions are normalized to sum to 1.
    """
    k = expected_num_splits(distance_m)
    if k <= 0:
        raise ValueError("Distance must be a positive multiple of 50.")

    models, metas = load_frac_models(stroke)

    preds = []
    for i in range(1, k + 1):
        t = frac_name(i)
        if t not in models:
            break
        pipe = models[t]
        feats = metas[t].get("features", [])
        X = build_feature_row(feats, distance_m, total_time_s)
        y = float(pipe.predict(X)[0])
        preds.append((t, max(0.0, y)))

    if not preds:
        raise RuntimeError(f"No usable frac models for {stroke} at {distance_m}m.")

    ser = pd.Series(dict(preds))
    total = ser.sum()
    if total > 0:
        ser = ser / total
    else:
        ser[:] = 1.0 / len(ser)
    return ser


def ideal_splits_seconds(stroke: str, distance_m: int, total_time_s: float) -> List[float]:
    """Convert fractions -> per-50 splits in seconds."""
    fracs = predict_fractions(stroke, distance_m, total_time_s)
    return [float(f * total_time_s) for f in fracs.values]


# ------------------------------------------------------------
# ANALYSIS + PLOTTING HELPERS
# ------------------------------------------------------------
def compare_df(given: List[float], ideal: List[float]) -> pd.DataFrame:
    n = min(len(given), len(ideal))
    df = pd.DataFrame(
        {
            "Distance (m)": [i * 50 for i in range(1, n + 1)],
            "Given (s)": [float(x) for x in given[:n]],
            "Ideal (s)": [float(x) for x in ideal[:n]],
        }
    )
    df["Delta (Given - Ideal)"] = df["Given (s)"] - df["Ideal (s)"]
    return df


def plot_lines(x, y_list, labels, title, xlabel, ylabel, save_jpeg_path: str = None):
    plt.figure(figsize=(8, 4))
    for y, lab in zip(y_list, labels):
        plt.plot(x, y, marker="o", label=lab)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    if save_jpeg_path:
        plt.tight_layout()
        plt.savefig(save_jpeg_path, dpi=200, format="jpeg")
        print(f"Saved chart: {save_jpeg_path}")
    plt.show()


# ------------------------------------------------------------
# SMALL CLI HELPERS
# ------------------------------------------------------------
def pick_from_list(prompt: str, options: List[str]) -> str:
    print(prompt)
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")
    while True:
        s = input("> ").strip()
        if s.isdigit() and 1 <= int(s) <= len(options):
            return options[int(s) - 1]
        for opt in options:
            if s.lower() == opt.lower():
                return opt
        print("Choose a valid option.")


def ask_int(prompt: str) -> int:
    while True:
        s = input(prompt).strip()
        try:
            return int(s)
        except:
            print("Please enter an integer.")


def ask_time_seconds(prompt: str) -> float:
    while True:
        s = input(prompt).strip()
        v = parse_time_to_seconds(s)
        if np.isfinite(v):
            return v
        print("Please enter a valid time (e.g., 59.80 or 1:05.23).")


def ask_yes_no(prompt: str) -> bool:
    s = input(prompt + " [y/n]: ").strip().lower()
    return s in ("y", "yes")


def ask_save_path(default_name: str) -> str:
    s = input(f"Enter JPEG filename (or press Enter for '{default_name}'): ").strip()
    if not s:
        s = default_name
    if not s.lower().endswith(".jpg") and not s.lower().endswith(".jpeg"):
        s += ".jpg"
    return s


# ------------------------------------------------------------
# CLI MODES
# ------------------------------------------------------------
def cli_pre_race():
    print("\n=== PRE-RACE: Ideal Splits ===")
    stroke = pick_from_list("Select stroke:", STROKES)
    distance = ask_int("Enter distance (e.g., 100, 200, 400): ")
    target_sec = ask_time_seconds("Enter TARGET time: ")

    try:
        ideal = ideal_splits_seconds(stroke, distance, target_sec)
    except Exception as e:
        print(f"Prediction failed: {e}")
        return

    print("\nIdeal splits (s per 50m):")
    print(print_splits(ideal))
    print(f"Target total: {seconds_to_time_str(target_sec)}")

    x = [i * 50 for i in range(1, len(ideal) + 1)]
    title = f"Ideal Splits — {stroke} {distance}m (target {seconds_to_time_str(target_sec)})"

    save_path = None
    if ask_yes_no("Save chart as JPEG?"):
        save_path = ask_save_path(f"ideal_{stroke}_{distance}m.jpg")

    plot_lines(x, [ideal], ["Ideal"], title, "Distance (m)", "Split (s)", save_path)


def cli_post_race():
    print("\n=== POST-RACE: Given vs Ideal ===")
    stroke = pick_from_list("Select stroke:", STROKES)
    distance = ask_int("Enter distance (e.g., 100, 200, 400, 800, 1500): ")
    pb = ask_time_seconds("Enter PERSONAL BEST: ")

    splits_str = input(
        'Paste race splits per 50 separated by ";" (e.g., 32.33; 33.22; 34.10):\n> '
    )
    given = parse_splits_string(splits_str)
    if not given:
        print("No numeric splits detected.")
        return

    total_actual = float(np.sum(given))

    try:
        ideal = ideal_splits_seconds(stroke, distance, total_actual)
    except Exception as e:
        print(f"Prediction failed: {e}")
        return

    print("\nGiven splits (s):")
    print(print_splits(given))
    print("\nIdeal splits (s):")
    print(print_splits(ideal))
    print(f"\nActual total: {seconds_to_time_str(total_actual)} | PB: {seconds_to_time_str(pb)}")

    df = compare_df(given, ideal)
    display(df)

    x = df["Distance (m)"].tolist()
    title = f"{stroke} {distance}m — Given vs Ideal (total {seconds_to_time_str(total_actual)})"

    save_path = None
    if ask_yes_no("Save comparison chart as JPEG?"):
        save_path = ask_save_path(f"compare_{stroke}_{distance}m.jpg")

    plot_lines(
        x,
        [df["Given (s)"].tolist(), df["Ideal (s)"].tolist()],
        ["Given", "Ideal"],
        title,
        "Distance (m)",
        "Split (s)",
        save_path,
    )


# ------------------------------------------------------------
# MAIN CLI LOOP
# ------------------------------------------------------------
def main():
    print("\nCompetitive Pacing — Simple CLI")
    print("--------------------------------")
    print("1) Pre-race (ideal splits)")
    print("2) Post-race (compare given vs ideal)")
    print("q) Quit")

    while True:
        choice = input("\nChoose 1 / 2 / q: ").strip().lower()
        if choice == "1":
            cli_pre_race()
        elif choice == "2":
            cli_post_race()
        elif choice in {"q", "quit", "exit"}:
            print("Bye.")
            break
        else:
            print("Please choose 1, 2, or q.")


# Kick off CLI after training
main()
